# RealHiTBench 수치추론 771건 — 큰 표에서 검색이 표-제공을 이기는가

**사전 등록:** `PREREG-2026-08-27-rhb-nr-large-tables.md`. 예측 P1~P8이 실행 **전에**
숫자로 박혀 있고, 이 노트북은 그것을 실행할 뿐 규칙을 고르지 않는다.

**주장:** 표가 리더의 유효 독해 용량을 넘으면, 검색으로 뽑은 셀 몇 줄이 표 전체보다 낫다.
무대가 맞는 이유는 NR gold 표가 **중앙값 1,300토큰 · 82%가 512 예산 초과 · 최대 12,477**이기 때문.

| arm | 내용 |
|---|---|
| `cell` | 코퍼스 전체 셀 색인 → 예산까지 (**본 방법**) |
| `goldtable` | **gold 표를 예산 무시하고 통째로** = RealHiTBench 논문의 조건 재현 |
| `flat` | 잎 라벨만 (통제) |

**리더 둘을 돌린다.** 약한 범용 7B(발표 NR 5.32)와 표 특화 7B(발표 29.31).
P8 — 두 리더 모두에서 P1이 서는가 — 가 이 실험의 진짜 시험이다. 약한 리더에서만 서면
주장은 "리더가 나빠서"로 반박당한다.

> Colab GPU는 **API가 아니다.** 가중치를 직접 받아 이 프로세스에서 돌리므로 `local:` 스펙
> 그대로고, "리더는 로컬" 제약을 깨지 않는다. 빌린 것은 GPU뿐이다.

**셀을 위에서부터 순서대로.** 세션이 끊기면 1~4를 다시 돌린 뒤 5번부터 이어가면 된다
(resume 내장 — 같은 `--out`을 주면 기록된 질의는 건너뛴다).


## 1. GPU 확인 — dtype이 여기서 정해진다

런타임 → 런타임 유형 변경 → **T4 GPU**.


In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())
assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다: 런타임 유형 변경 → T4 GPU'
GB  = torch.cuda.get_device_properties(0).total_memory / 1024**3
CAP = torch.cuda.get_device_capability(0)
print(f'VRAM {GB:.1f} GB | compute capability {CAP[0]}.{CAP[1]}')

# Turing(7.5, T4)에는 bfloat16 하드웨어가 없다. 로더 기본값이 bfloat16이므로
# 반드시 넘겨야 한다 -- 안 넘기면 느려지고, 기록된 리더 이름은 같아 보인다.
DTYPE = 'bfloat16' if CAP[0] >= 8 else 'float16'
assert GB >= 14, f'4bit 7B 두 개와 12k 토큰 문맥에는 14GB 이상 필요 (지금 {GB:.1f}GB)'
print(f'dtype={DTYPE}')


TOKEN  = ''   # 비공개 저장소면 GitHub PAT
BRANCH = 'fix/encoder-provenance'

import os, pathlib, subprocess
os.chdir('/content')  # rm -rf 후 CWD가 사라졌을 수 있다
url = f'https://{TOKEN}@github.com/Jax0303/T2.git' if TOKEN else 'https://github.com/Jax0303/T2.git'
if not pathlib.Path('/content/T2').exists():
    r = subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, url, '/content/T2'],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'git clone 실패 (exit {r.returncode}):\n{r.stderr}\n'
                           f'TOKEN이 비어있으면 비공개 저장소는 clone 불가')
assert pathlib.Path('/content/T2/rag-agent').exists(), '/content/T2/rag-agent 가 없습니다 — clone을 확인하세요'
os.chdir('/content/T2/rag-agent')
git push origin fix/encoder-provenance log --oneline -1

# torch는 Colab에 이미 있다. 나머지만. flash-attn은 T4(sm_75)를 지원하지 않으므로 설치하지 않는다.
pip install torch torchvision numpy pandas pillow matplotlib tqdm --break-system-packages -q install 'transformers>=4.40' 'bitsandbytes>=0.43' 'accelerate' \\n                'sentence-transformers>=3.0' 'faiss-cpu>=1.7' 'rank-bm25==0.2.2' \\n                'scipy>=1.11' 'scikit-learn>=1.4' 'tabulate>=0.9' 2>&1 | tail -2
print('done')


In [ ]:
TOKEN  = ''   # 비공개 저장소면 GitHub PAT
BRANCH = 'fix/encoder-provenance'

import os, pathlib
url = f'https://{TOKEN}@github.com/Jax0303/T2.git' if TOKEN else 'https://github.com/Jax0303/T2.git'
if not pathlib.Path('/content/T2').exists():
    !git clone --depth 1 -b {BRANCH} {url} /content/T2
os.chdir('/content/T2/rag-agent')
!git log --oneline -1

# torch는 Colab에 이미 있다. 나머지만. flash-attn은 T4(sm_75)를 지원하지 않으므로 설치하지 않는다.
!pip -q install 'transformers>=4.40' 'bitsandbytes>=0.43' 'accelerate' \
                'sentence-transformers>=3.0' 'faiss-cpu>=1.7' 'rank-bm25==0.2.2' \
                'scipy>=1.11' 'scikit-learn>=1.4' 'tabulate>=0.9' 2>&1 | tail -2
print('done')


## 3. 데이터 + 결과 저장소 (Drive)

**RealHiTBench는 저장소에 없고(`.gitignore`), 공식 배포로 대체할 수도 없다.**
동결된 모집단 `rhb_nr_all`(n=764)은 이 저장소가 쓰는 **로컬 사본**(3,071 QA / HTML 540표)에서
유도됐다. 공식 배포(3,752 QA / 708표)를 받으면 id가 어긋나 `pin`이 실행을 거부한다 —
그게 정상 동작이다. 그러니 로컬 사본을 Drive로 한 번 올린다.

```bash
# 연구실 기계에서 한 번:
cd ~/T2/rag-agent && tar czf ~/rhb_data.tar.gz data/realhitbench
# 그 파일을 Google Drive의  MyDrive/T2_rhbnr/rhb_data.tar.gz  에 올린다 (7.6MB — HTML이라 잘 압축된다)
```

결과도 같은 폴더에 쌓는다. Colab이 죽으면 `/content`는 사라지지만 Drive는 남고,
다음 세션이 그걸 되살려 **중단 지점부터** 이어간다.


In [ ]:
from google.colab import drive
import pathlib, shutil, tarfile
drive.mount('/content/drive')
SAVE = pathlib.Path('/content/drive/MyDrive/T2_rhbnr')
SAVE.mkdir(parents=True, exist_ok=True)

# 데이터 (세션당 한 번)
if not pathlib.Path('data/realhitbench/QA_final.json').exists():
    tar = SAVE / 'rhb_data.tar.gz'
    assert tar.exists(), f'{tar} 가 없습니다 — 위 markdown의 tar 명령을 먼저 실행해 올리세요'
    tarfile.open(tar).extractall('.')
print('QA:', len(__import__('json').load(open('data/realhitbench/QA_final.json'))['queries']),
      '| html:', len(list(pathlib.Path('data/realhitbench/html').glob('*.html'))))

# 지난 세션 결과 복원 -> resume이 이걸 읽는다
pathlib.Path('results').mkdir(exist_ok=True)
for f in SAVE.glob('rhbnr_*'):
    shutil.copy(f, 'results/' + f.name)
print('복원:', sorted(f.name for f in SAVE.glob('rhbnr_*')) or '(없음)')


## 4. 모집단 확인 — 764개가 나와야 한다

`rhb_nr_all`은 저장소에 동결돼 있다. 여기서 재유도해 **드리프트만** 본다.
숫자가 다르면 파서나 데이터 사본이 달라진 것이므로, 그대로 진행하면 안 된다.


In [ ]:
!PYTHONPATH=.:scripts python scripts/freeze_populations.py --only rhb_nr_all --check 2>&1 | tail -5
!head -3 populations/rhb_nr_all.txt


## 4.5 처리율 보정 — 본 실행 전에 2분만

아래 전체 시간 추정은 **prefill 500 tok/s · decode 12 tok/s**를 가정한 것이다.
T4 실측이 이와 다르면 추정이 통째로 틀린다. 리더를 한 번 띄워 짧은/긴 프롬프트를
각각 재고, 남은 시간을 다시 계산한다. 여기서 잰 리더는 5번 셀이 그대로 재사용한다.


In [ ]:
import time, torch
from rag_agent.llm.factory import build_llm

llm = build_llm(f'local:Qwen/Qwen2.5-7B-Instruct?quantization=4bit&dtype={DTYPE}')
tok = llm.tokenizer

def bench(ctx_tokens, out_tokens, reps=3):
    """prefill/decode를 분리해 재려면 두 길이가 필요하다 -- 한 점으로는 못 가른다."""
    filler = 'row header | col header: 123 . ' * (ctx_tokens // 8 + 1)
    user = filler[:len(filler)] + '\n\nQUESTION: what is the value?'
    user = tok.decode(tok(user).input_ids[:ctx_tokens])
    ts = []
    for _ in range(reps):
        t0 = time.time(); llm.complete(system='Answer with the value only.', user=user,
                                       max_tokens=out_tokens); ts.append(time.time() - t0)
    return min(ts)                      # 최소값: 웜업/스케줄링 잡음 제거

_ = bench(256, 4, reps=1)               # 워밍업 (첫 호출은 항상 느리다)
t_short, t_long = bench(256, 8), bench(2048, 8)
t_out          = bench(256, 64)

# 두 컨텍스트 길이의 차이가 순수 prefill, 출력 길이의 차이가 순수 decode
PREFILL = (2048 - 256) / max(t_long - t_short, 1e-6)
DECODE  = (64 - 8) / max(t_out - t_short, 1e-6)
print(f'실측  prefill {PREFILL:6.0f} tok/s   decode {DECODE:5.1f} tok/s')
print(f'가정  prefill    500 tok/s   decode  12.0 tok/s')

# RHB NR 764문항의 실측 컨텍스트: cell 511 / flat 511 / goldtable 평균 1839
OVERHEAD, ARMS_CTX = 260, (511, 511, 1839)
for mode, out in (('direct', 8), ('codegen', 40)):
    sec = sum(764 * ((c + OVERHEAD) / PREFILL + out / DECODE) for c in ARMS_CTX)
    print(f'  {mode:8} 764문항 x 3 arm  ->  {sec/3600:5.2f} 시간')


## 5. 리더 A — 약한 범용 7B (발표 NR **5.32**)

764문항 × 3 arm. T4에서 대략 **1.5~2.5시간**. `goldtable` arm이 문맥을 예산 없이
통째로 넣기 때문에 (평균 1,839토큰, 최대 12,477) 이 arm이 시간의 절반을 쓴다.

끊기면 이 셀만 다시 실행하면 된다 — 이미 기록된 질의는 건너뛴다.


In [ ]:
BUDGET = 512
ARMS   = 'cell,goldtable,flat'
READER_A = f'local:Qwen/Qwen2.5-7B-Instruct?quantization=4bit&dtype={DTYPE}'

!PYTHONPATH=.:scripts python scripts/corpus_dump_vs_cell.py \
  --dataset realhitbench --population rhb_nr_all \
  --arms {ARMS} --cell-scheme S3c --retriever dense --budget {BUDGET} \
  --reader "{READER_A}" \
  --out results/rhbnr_s3c_{BUDGET}.json 2>&1 | tail -25


In [ ]:
# 끝나는 대로 Drive에 (중간에 끊겨도 이 셀만 돌리면 진행분이 보존된다)
for f in pathlib.Path('results').glob(f'rhbnr_s3c_{BUDGET}*'):
    shutil.copy(f, SAVE / f.name)
print('저장:', sorted(f.name for f in SAVE.glob('rhbnr_*')))


## 6. 리더 B — 표 특화 7B (발표 NR **29.31**)

**바꾸는 것은 리더 하나뿐이다.** 모집단·검색·예산·arm·채점기 전부 동일.
같은 파라미터 수인데 발표 수치가 5.5배다 — 표를 통째로 읽는 능력 자체가 A보다 훨씬 낫다.
그런데도 큰 표에서 검색이 이기면, 주장은 리더 품질이 아니라 **문맥 길이**에 관한 것이 된다.

> 4-bit로 안 올라가면 사전등록 §4의 규칙대로 **A만으로 보고하고 P8은 미측정으로 남긴다.**
> A 결과를 P8이 선 것처럼 서술하지 않는다.


In [ ]:
READER_B = f'local:tablegpt/TableGPT2-7B?quantization=4bit&dtype={DTYPE}'

!PYTHONPATH=.:scripts python scripts/corpus_dump_vs_cell.py \
  --dataset realhitbench --population rhb_nr_all \
  --arms {ARMS} --cell-scheme S3c --retriever dense --budget {BUDGET} \
  --reader "{READER_B}" \
  --out results/rhbnr_tgpt_{BUDGET}.json 2>&1 | tail -25


In [ ]:
for f in pathlib.Path('results').glob(f'rhbnr_tgpt_{BUDGET}*'):
    shutil.copy(f, SAVE / f.name)
print('저장:', sorted(f.name for f in SAVE.glob('rhbnr_*')))


## 7. 판정 — 사전등록 P1~P8

산술이지 서술이 아니다. `scripts/rhbnr_verdict.py`가 예측 구간과 실측을 대조한다.

| | 예측 | 구간 |
|---|---|---|
| **P1 (주)** | gold 표 >2048토큰에서 `cell` − `goldtable` | ≥ +.05 |
| **P2 (주)** | gold 표 ≤512토큰에서 `goldtable` − `cell` | ≥ +.03 |
| **P3** | P1·P2 부호가 반대 (교차점) | 부호만 |
| **P4** | 771 전수 `cell` EM | .03 ~ .08 |
| **P5** | `cell` > `flat` | ≥ +.03 |
| **P8 (진짜 시험)** | P1이 **두 리더 모두**에서 성립 | 부호만 |


In [ ]:
B_ARG = f'--b results/rhbnr_tgpt_{BUDGET}_records.jsonl' if pathlib.Path(f'results/rhbnr_tgpt_{BUDGET}_records.jsonl').exists() else ''
!PYTHONPATH=.:scripts python scripts/rhbnr_verdict.py \
  --a results/rhbnr_s3c_{BUDGET}_records.jsonl {B_ARG} \
  --out results/rhbnr_verdict.json
shutil.copy('results/rhbnr_verdict.json', SAVE / 'rhbnr_verdict.json')


## 8. 다음 — 판정 규칙 (사전등록 §4, 결과를 보고 고르지 않는다)

| 결과 | 다음 할 일 |
|---|---|
| P1·P2 둘 다 섬 | **프레임 확정.** 논문에 "표 크기 조건부로 검색이 표-제공을 이긴다"를 쓴다 |
| P1만 섬 | 교차점 주장은 버리고 "큰 표에서 검색이 낫다"만 쓴다 |
| P1이 안 섬 | **프레임 폐기.** RHB 수치추론은 이 방법의 무대가 아니라고 적는다 |
| P8이 섬 | 주장은 리더 품질이 아니라 **문맥 길이**에 관한 것이다 (최선의 결과) |
| P8이 깨짐 | "약한 리더 한정"으로 범위를 좁혀 적는다. **숨기지 않는다** |

⚠️ **P7 — 발표 수치 5.32와의 대조는 불확실하다고 실행 전에 적어뒀다.** P4의 구간이
5.32를 가로지른다. 넘으면 보고하고, **못 넘어도 P1이 서면 주장은 산다** — 주장은
"발표 수치를 이겼다"가 아니라 "표가 크면 검색이 표-제공을 이긴다"이기 때문이다.

⚠️ **로컬 사본 ≠ 공식 배포.** 논문에 어느 쪽을 썼는지 명시한다
(로컬 3,071 QA / HTML 540표 · 공식 708표 / 3,752 QA).
